# 03 — BOW, DTM, TFIDF, TFIDF_L2
Builds all vectorized representations of the corpus.

**Inputs:** `data/docs_sampled.csv`, `data/hc3_CORPUS.csv`, `data/hc3_VOCAB.csv`

**Outputs:** `data/hc3_BOW.csv`, `data/hc3_DTM.csv`, `data/hc3_TFIDF.csv`, `data/hc3_TFIDF_L2.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [2]:
docs_sampled = pd.read_csv('../data/docs_sampled.csv')
CORPUS= pd.read_csv('../data/hc3_CORPUS.csv')
VOCAB = pd.read_csv('../data/hc3_VOCAB.csv').set_index('term_str')

CORPUS_reset = CORPUS
doc_texts= docs_sampled.set_index('doc_id')['text']

print(f'docs_sampled: {len(docs_sampled):,}')
print(f'CORPUS: {len(CORPUS):,}')
print(f'VOCAB: {len(VOCAB):,}')

docs_sampled: 10,258
CORPUS: 1,607,435
VOCAB: 40,531


## BOW — Bag of Words

In [3]:
BOW = (
    CORPUS_reset
    .groupby(['doc_id','term_str'])
    .size()
    .rename('n')
    .reset_index()
)

N= CORPUS_reset['doc_id'].nunique()
df_map = VOCAB['df'].to_dict()
BOW['tf']= BOW.groupby('doc_id')['n'].transform(lambda x: x / x.sum())
BOW['idf'] = BOW['term_str'].map(lambda t: np.log(N / (df_map.get(t, 1))))
BOW['tfidf'] = BOW['tf'] * BOW['idf']

print(f'BOW shape: {BOW.shape}')
print(f'Columns: {list(BOW.columns)}')
BOW.head()

BOW shape: (828447, 6)
Columns: ['doc_id', 'term_str', 'n', 'tf', 'idf', 'tfidf']


,doc_id,term_str,n,tf,idf,tfidf
0,10001_AI0,'s,2,0.008299,0.815131,0.006765
1,10001_AI0,(,1,0.004149,1.025145,0.004254
2,10001_AI0,),1,0.004149,1.015410,0.004213
3,10001_AI0,",",17,0.070539,0.130833,0.009229
4,10001_AI0,.,9,0.037344,0.029380,0.001097


In [4]:
BOW.to_csv('../data/hc3_BOW.csv', index=False)
print('Saved data/hc3_BOW.csv')

Saved data/hc3_BOW.csv


## DTM — Document Term Matrix (count)

In [5]:
import string
sig_terms= [
    t for t in VOCAB[~VOCAB['stop']].head(6000).index.tolist()
    if t.isalpha() and t not in string.punctuation
][:5000]

cv= CountVectorizer(vocabulary=sig_terms, lowercase=True, token_pattern=r'\b[a-z]+\b')
DTM_matrix = cv.fit_transform(doc_texts)

DTM = pd.DataFrame.sparse.from_spmatrix(
    DTM_matrix,
    index=doc_texts.index,
    columns=cv.get_feature_names_out()
)
print(f'DTM shape: {DTM.shape} (documents x terms)')

DTM shape: (10258, 5000) (documents x terms)


In [6]:
DTM.sparse.to_dense().to_csv('../data/hc3_DTM.csv')
print('Saved data/hc3_DTM.csv')

Saved data/hc3_DTM.csv


## TFIDF — TF-IDF Matrix

In [7]:
tv = TfidfVectorizer(
    vocabulary=sig_terms,lowercase=True,
    token_pattern=r'\b[a-z]+\b',
    use_idf=True,smooth_idf=True, sublinear_tf=False
)
TFIDF_matrix = tv.fit_transform(doc_texts)

TFIDF = pd.DataFrame.sparse.from_spmatrix(
    TFIDF_matrix,
    index=doc_texts.index,
    columns=tv.get_feature_names_out()
)
print(f'TFIDF shape: {TFIDF.shape}')
print('Formula: tf(t,d) * log((1+N)/(1+df(t))) + 1  [sklearn smooth idf]')

TFIDF shape: (10258, 5000)
Formula: tf(t,d) * log((1+N)/(1+df(t))) + 1  [sklearn smooth idf]


In [8]:
TFIDF.sparse.to_dense().to_csv('../data/hc3_TFIDF.csv')
print('Saved data/hc3_TFIDF.csv')

Saved data/hc3_TFIDF.csv


## TFIDF_L2 — Reduced and L2 Normalized

In [9]:
N_FEATURES = 2000
top_terms= [
    t for t in VOCAB[~VOCAB['stop']].head(N_FEATURES * 2).index.tolist()
    if t.isalpha() and t not in string.punctuation
][:N_FEATURES]
top_terms = [t for t in top_terms if t in TFIDF.columns]

TFIDF_reduced = TFIDF[top_terms].sparse.to_dense()
TFIDF_L2 = pd.DataFrame(
    normalize(TFIDF_reduced.values, norm='l2'),
    index=TFIDF_reduced.index,
    columns=TFIDF_reduced.columns
)
print(f'TFIDF_L2 shape: {TFIDF_L2.shape}')
print(f'Features: top {N_FEATURES} non-stop words by DFIDF, L2 normalized')

TFIDF_L2 shape: (10258, 2000)
Features: top 2000 non-stop words by DFIDF, L2 normalized


In [10]:
TFIDF_L2.to_csv('../data/hc3_TFIDF_L2.csv')
print('Saved data/hc3_TFIDF_L2.csv')

Saved data/hc3_TFIDF_L2.csv
